# VERA — RL Fine-Tuning (REINFORCE)

Runs **Phase 2 RL training** starting from an SFT checkpoint, using `rl_trainer_vera.py`.

**Loss**: REINFORCE + value baseline + entropy bonus + KL penalty vs frozen BC anchor  
**Expected time**: ~2–4 hrs for 100 epochs on T4 GPU (Language-Table)

---

## Before running
1. `Runtime → Change runtime type → T4 GPU`
2. Upload your SFT checkpoint (`best_sft_vera.pt`) to Drive at:
   ```
   MyDrive/VERA_LT_Checkpoints/rl_seed123/best_sft_vera.pt
   ```
   *(rename the file from your seed123 SFT run — it goes in the **rl_seed123** folder)*
3. Run all cells top-to-bottom

## Output
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/best_rl_vera.pt` — best checkpoint by return
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/rl_vera_log.json` — full epoch log
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/sample_efficiency.csv` — for plotting
- Live epoch-by-epoch output in cell 6 below

## Running multiple seeds
Open separate Colab tabs and change `SEED` in Cell 4 (e.g. `SEED = 42`, `SEED = 456`).  
Each seed needs its own SFT checkpoint folder:  
`rl_seed42/best_sft_vera.pt`, `rl_seed456/best_sft_vera.pt`

In [1]:
# ── Cell 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# ── Cell 2: Clone repo (or use Drive copy) ─────────────────────────────────
import os, sys

REPO_DRIVE = '/content/drive/MyDrive/VLA-Robot-Learning'
REPO_LOCAL = '/content/RLConditionedVLA'   # matches GitHub repo name

if os.path.isdir(REPO_DRIVE):
    print(f'Repo found on Drive at {REPO_DRIVE}')
    REPO = REPO_DRIVE
elif os.path.isdir(REPO_LOCAL):
    print(f'Repo already cloned at {REPO_LOCAL}')
    REPO = REPO_LOCAL
else:
    print('Cloning repo from GitHub...')
    os.system(f'git clone https://github.com/sara-kaz/RLConditionedVLA.git {REPO_LOCAL}')
    REPO = REPO_LOCAL
    print(f'Cloned to {REPO}')

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(f'Using repo: {REPO}')
print(f'Config path: {REPO}/configs/config.yaml')

In [ ]:
# ── Cell 2b: Pull latest code from GitHub ──────────────────────────────────
# Run this after Cell 2 to make sure you have the newest fixes.
# Critical: LanguageTableEnv fix (commit 4af2aea) must be present.
import subprocess, os

result = subprocess.run(
    ['git', '-C', REPO, 'pull', '--rebase', 'origin', 'main'],
    capture_output=True, text=True
)
print(result.stdout.strip() or '(nothing to pull)')
if result.returncode != 0:
    print('Pull failed — trying fresh clone:')
    os.system(f'rm -rf {REPO}')
    os.system(f'git clone https://github.com/sara-kaz/RLConditionedVLA.git {REPO}')

# Verify the Language-Table env fix is present
with open(f'{REPO}/envs/sim_env.py') as f:
    content = f.read()
if 'LanguageTableEnv' in content:
    print('✓ LanguageTableEnv class present in envs/sim_env.py')
else:
    raise RuntimeError(
        'LanguageTableEnv NOT found — fresh clone failed. '
        'Check your GitHub connection.'
    )

with open(f'{REPO}/training/rl_trainer_vera.py') as f:
    trainer = f.read()
if 'make_env' in trainer:
    print('✓ make_env routing present in rl_trainer_vera.py')
else:
    raise RuntimeError('Trainer still calls SimEnv directly — pull failed.')

In [ ]:
# ── Cell 3: Install dependencies ───────────────────────────────────────────
import subprocess, sys, os

def pip(*pkgs, no_deps=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if no_deps:
        cmd.append('--no-deps')
    cmd.extend(pkgs)
    subprocess.check_call(cmd)

# Base deps
pip('ftfy', 'regex', 'tqdm', 'pyyaml', 'pillow', 'numpy')

# gym version compatible with language-table
pip('gym<=0.23.0')
pip('pybullet')

# dm-env: language-table is built on DeepMind's env library (not gym)
pip('dm-env')

# language-table must be installed --no-deps from GitHub (not on PyPI)
pip('git+https://github.com/google-research/language-table.git', no_deps=True)

# CLIP
try:
    import clip
    print('CLIP already installed.')
except ImportError:
    pip('git+https://github.com/openai/CLIP.git')

# ── Headless display for pybullet / Language-Table ─────────────────────────
print('Setting up virtual display (Xvfb) for Language-Table / pybullet …')
subprocess.run(['apt-get', 'install', '-y', '-q', 'xvfb'], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.environ['DISPLAY'] = ':99'
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1024x768x24', '-ac'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import time; time.sleep(1)
print(f'  DISPLAY={os.environ["DISPLAY"]}  ✓')

# Verify Language-Table import and probe available attributes
from language_table.environments import language_table as _lt_check
import language_table as _lt_top
print(f'language_table import OK')
print(f'  Top-level attrs: {[a for a in dir(_lt_top) if not a.startswith("_")]}')
print(f'  lt_module attrs: {[a for a in dir(_lt_check) if not a.startswith("_")]}')
print('All dependencies installed.')

In [ ]:
# ── Cell 4: ⚙️  USER CONFIG ────────────────────────────────────────────────
import os, glob

MYDRIVE = '/content/drive/MyDrive'
SEED    = 123   # ← change to 42 or 456 for other seeds

# ── RL hyperparameters (Round 4 — EXACT Round 1 rerun + save_every=1) ─────
# Root-cause analysis of all rounds:
#   Round 1: DAgger, rollouts=8,  lr=3e-5, kl=0.20 → 50% peak (ep35) ✓ BEST
#            PROBLEM: best_sr_vera.pt didn't exist — peak was never saved.
#   Round 2: RL return-best start (wrong!), rollouts=16, kl=0.10 → 37.5%
#   Round 3: DAgger (correct), rollouts=16, kl=0.05 → 25%
#            PROBLEM: kl=0.05 too low (policy drifted from DAgger anchor);
#                     rollouts=16 changed gradient landscape vs Round 1.
# Round 4 fix: replicate Round 1 EXACTLY — the only proven config.
#   Change vs Round 1: save_every=1 (catch the ep35 peak this time)
RL_EPOCHS          = 50        # Round 1 peaked at ep35 — 50 gives margin
RL_NUM_ROLLOUTS    = 8         # ← back to Round 1's 8  (16 hurt in Round 3)
RL_MAX_EP_STEPS    = 60
RL_LR              = 3e-5      # Round 1's proven lr — do NOT change
RL_ENTROPY_COEF    = 0.05      # Round 1's value — do NOT change
RL_KL_COEF         = 0.20      # ← back to Round 1's 0.20 (0.05 let policy drift → 25%)
RL_VF_COEF         = 0.5
RL_GAMMA           = 0.99
RL_GRAD_CLIP       = 1.0
RL_SR_PATIENCE     = 15        # early-stop 15 epochs after last SR improvement
FREEZE_CLIP          = True
UNFREEZE_CLIP_VISION = False

# ── Language-Table data path ───────────────────────────────────────────────
LT_DATA_PATH = f'{MYDRIVE}/VERA_LT_Real/lt_real_data'

# ── Checkpoint paths ───────────────────────────────────────────────────────
DAGGER_CKPT_DIR = f'{MYDRIVE}/VERA_LT_Real/checkpoints/lt_full_vera_rl/seed{SEED}'
RL_OUT_DIR      = f'{DAGGER_CKPT_DIR}/rl'
os.makedirs(RL_OUT_DIR, exist_ok=True)

# Round 4: ALWAYS start from DAgger best_v3.pt — Round 1 proved this gives 50%.
# The RL checkpoints from Rounds 2-3 (37.5% and 25%) are WORSE starting points.
ROUND4_CANDIDATES = [
    f'{DAGGER_CKPT_DIR}/best_v3.pt',          # ← DAgger best (FIRST — Round 1's proven start)
    f'{DAGGER_CKPT_DIR}/latest_v6.pt',         # ← DAgger latest fallback
    # Do NOT use RL checkpoints — they are degraded relative to DAgger
]

print(f'Seed {SEED} — scanning for Round 4 starting checkpoint:\n')
SFT_CKPT = ''
for path in ROUND4_CANDIDATES:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        tag = ' ← SELECTED' if not SFT_CKPT else ''
        print(f'  ✓ {path}  ({size_mb:.0f} MB){tag}')
        if not SFT_CKPT:
            SFT_CKPT = path
    else:
        print(f'  ✗ {path}')

if not SFT_CKPT:
    raise FileNotFoundError(
        'DAgger best_v3.pt not found on Drive!\n'
        f'Expected: {DAGGER_CKPT_DIR}/best_v3.pt\n'
        'Upload your DAgger checkpoint before running Round 4.'
    )

print(f'\n✓ Starting from DAgger checkpoint (Round 1 config = 50% peak at epoch 35)')
print(f'LT data: {"✓ found" if os.path.isdir(LT_DATA_PATH) else "✗ NOT FOUND"}')

In [ ]:
# ── Cell 5: Write config + convert checkpoint to RL trainer format ──────────
import yaml, os, shutil, torch
from datetime import datetime

if not SFT_CKPT:
    raise ValueError('SFT_CKPT is empty — run Cell 4 first.')
if not os.path.exists(SFT_CKPT):
    raise FileNotFoundError(f'Checkpoint not found: {SFT_CKPT}')

# ── Back up any existing best_sr_vera.pt BEFORE this round overwrites it ──
# The trainer resets best_sr = -inf each run, so a worse round WILL overwrite
# a better previous checkpoint. Backup preserves it.
existing_sr = f'{RL_OUT_DIR}/rl/best_sr_vera.pt'
if os.path.exists(existing_sr):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    bak = f'{RL_OUT_DIR}/rl/best_sr_vera_backup_{ts}.pt'
    shutil.copy2(existing_sr, bak)
    try:
        meta = torch.load(bak, map_location='cpu')
        old_sr = meta.get('success_rate', '?')
        old_ep = meta.get('epoch', '?')
        print(f'⚠ Backed up previous best_sr_vera.pt → {bak}')
        print(f'  (was epoch={old_ep}, SR={old_sr if isinstance(old_sr, str) else f"{old_sr:.1%}"} — safe on Drive)')
    except Exception:
        print(f'⚠ Backed up previous best_sr_vera.pt → {bak}')

# ── Load DAgger checkpoint and re-save in RL trainer format ───────────────
print(f'\nLoading checkpoint: {SFT_CKPT}')
raw = torch.load(SFT_CKPT, map_location='cpu')

if isinstance(raw, dict):
    keys = list(raw.keys())
    print(f'  Checkpoint keys: {keys}')
    if 'model_state' in raw:
        model_sd = raw['model_state']
        print('  Format: RL trainer (model_state) — no conversion needed')
    elif 'model' in raw:
        model_sd = raw['model']
        sr = raw.get('best_sr', 0.0)
        ep = raw.get('epoch', '?')
        print(f'  Format: DAgger (model) — converting.  epoch={ep}, best_sr={sr:.1%}')
    else:
        model_sd = raw
        print('  Format: bare state dict')
else:
    raise TypeError(f'Unexpected checkpoint type: {type(raw)}')

SFT_LINK = f'{RL_OUT_DIR}/best_sft_vera.pt'
torch.save({'model_state': model_sd}, SFT_LINK)
print(f'  Saved converted checkpoint → {SFT_LINK}')

# ── Write patched config ───────────────────────────────────────────────────
CONFIG_PATH = f'{REPO}/configs/config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['rl']['epochs']            = RL_EPOCHS
cfg['rl']['num_rollouts']      = RL_NUM_ROLLOUTS
cfg['rl']['max_episode_steps'] = RL_MAX_EP_STEPS
cfg['rl']['lr']                = float(RL_LR)
cfg['rl']['entropy_coef']      = float(RL_ENTROPY_COEF)
cfg['rl']['kl_coef']           = float(RL_KL_COEF)
cfg['rl']['vf_coef']           = float(RL_VF_COEF)
cfg['rl']['gamma']             = float(RL_GAMMA)
cfg['rl']['grad_clip']         = float(RL_GRAD_CLIP)
cfg['rl']['sr_patience']       = int(RL_SR_PATIENCE)
cfg['rl']['save_every']        = 1       # save EVERY epoch — cannot miss the peak

cfg['model']['freeze_clip']          = FREEZE_CLIP
cfg['model']['unfreeze_clip_vision'] = UNFREEZE_CLIP_VISION
cfg['training']['seed']              = SEED
cfg['training']['output_dir']        = RL_OUT_DIR

# ── CRITICAL: set Language-Table env ─────────────────────────────────────────
# Previous rounds used env_id='dummy' → RandomDummyEnv (NOT Language-Table!).
# The SR numbers from Rounds 1-4 were measuring success on a random toy env.
# This line fixes the root cause — RL now rolls out in the real LT simulator.
cfg['env']['env_id']            = 'language_table'
cfg['env']['lt_velocity_scale'] = 0.03   # LT action space: [-0.03, 0.03] m/s

# Resolve "auto" device
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['training']['device'] = device_str
print(f'\nDevice: {device_str} ({"GPU ✓" if device_str == "cuda" else "CPU — switch to T4 GPU!"})')

if LT_DATA_PATH and os.path.isdir(LT_DATA_PATH):
    cfg['data']['episodes_path'] = LT_DATA_PATH
    cfg['data']['dataset_type']  = 'language_table'
    print(f'Data: Language-Table at {LT_DATA_PATH}')
else:
    cfg['data']['episodes_path'] = None
    cfg['data']['dataset_type']  = 'pkl'
    print('⚠️  No LT data — using synthetic (SR meaningless without real data)')

RL_CONFIG_PATH = f'{RL_OUT_DIR}/rl_config.yaml'
with open(RL_CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f)

print(f'\nConfig → {RL_CONFIG_PATH}')
print('RL settings:')
for k, v in cfg['rl'].items():
    print(f'  {k}: {v}')
print(f'\n★ env_id = {cfg["env"]["env_id"]}  ← rolling out in REAL Language-Table now')
print(f'  save_every=1 — checkpoint written every epoch')
print(f'  sr_patience={RL_SR_PATIENCE} — early stop after {RL_SR_PATIENCE} epochs without SR improvement')

In [ ]:
# ── Cell 5b: Smoke-test — verify Language-Table env loads ───────────────────
# Run this BEFORE Cell 6. If it prints "✓ Language-Table env working" you are
# rolling out in the real LT simulator. If you see LanguageTableEnv errors,
# fix them before starting the 50-epoch training run.
import sys, os
sys.path.insert(0, REPO)
os.environ.setdefault('DISPLAY', ':99')   # ensure Xvfb display is set

import yaml
with open(RL_CONFIG_PATH) as f:
    test_cfg = yaml.safe_load(f)

env_id = test_cfg.get('env', {}).get('env_id', 'NOT SET')
print(f'env_id in config: {env_id}')
if env_id != 'language_table':
    raise RuntimeError(f'env_id is "{env_id}", expected "language_table" — re-run Cell 5')

from envs.sim_env import make_env
print('Instantiating Language-Table environment (may take ~10s for pybullet) …')
try:
    env = make_env(test_cfg)
except Exception as e:
    raise RuntimeError(
        f'LanguageTableEnv failed to load: {e}\n\n'
        'Common fixes:\n'
        '  1. Check Cell 3 ran successfully (pybullet + language_table installed)\n'
        '  2. Check DISPLAY is set: run os.environ["DISPLAY"] = ":99" and restart Xvfb\n'
        '  3. Re-run Cell 2b to pull latest code'
    )

env_type = type(env).__name__
print(f'Env type: {env_type}')
assert 'LanguageTableEnv' in env_type, (
    f'Got {env_type} — still on dummy env! Pull latest code (Cell 2b) and re-run.'
)

# Run a quick 2-step episode to verify obs format and reward
obs = env.reset()
assert 'frame' in obs and 'instruction' in obs, f'Unexpected obs keys: {list(obs.keys())}'
print(f'Reset OK:')
print(f'  frame shape   : {obs["frame"].shape}  dtype={obs["frame"].dtype}')
print(f'  instruction   : "{obs["instruction"][:80]}"')

obs2, rew, done, info = env.step(0)   # action 0 = push right
print(f'Step  OK: reward={rew:.3f}  done={done}')
env.close()
print('\n✓ Language-Table env working — safe to run Cell 6 (real RL training)')

In [ ]:
# ── Cell 5c: DRIVE QUOTA FIX — run this if Drive is full ────────────────────
# save_every=1 across 50 epochs × ~50 MB = ~2.5 GB — that's what blew the quota.
# This cell:
#   1. Deletes redundant epoch snapshots from Drive to free space
#   2. Redirects this run's output to local /content/ disk (100 GB free on T4)
#   3. Re-saves the SFT checkpoint locally in case the Drive write failed
#   4. Patches the config to use local output + save_every=5
# After training, run Cell 6b to copy ONLY best_sr_vera.pt back to Drive.

import os, glob, shutil, torch, yaml

DRIVE_RL_DIR  = f'{DAGGER_CKPT_DIR}/rl/rl'   # where epoch checkpoints landed
LOCAL_OUT     = '/content/rl_run_seed123'      # local disk — plenty of space

# ── Step 1: Free Drive space ───────────────────────────────────────────────
print('Freeing Drive space (deleting redundant epoch snapshots) ...')
freed_bytes = 0
for pattern in [
    f'{DRIVE_RL_DIR}/rl_vera_epoch*.pt',          # per-epoch snapshots (all redundant)
]:
    matches = glob.glob(pattern)
    for path in sorted(matches):
        try:
            sz = os.path.getsize(path)
            os.remove(path)
            freed_bytes += sz
            print(f'  deleted {os.path.basename(path)}  ({sz/1e6:.0f} MB)')
        except Exception as e:
            print(f'  could not delete {path}: {e}')
if freed_bytes == 0:
    print('  (no epoch snapshots found — Drive may have refused writes when full)')
print(f'  Freed: {freed_bytes/1e6:.0f} MB from Drive')

# ── Step 2: Local output directory ────────────────────────────────────────
os.makedirs(f'{LOCAL_OUT}/rl', exist_ok=True)
print(f'\nLocal output dir: {LOCAL_OUT}  ({shutil.disk_usage("/content").free // 1e9:.0f} GB free)')

# ── Step 3: Ensure SFT checkpoint is on local disk ─────────────────────────
sft_drive = f'{RL_OUT_DIR}/best_sft_vera.pt'
sft_local = f'{LOCAL_OUT}/best_sft_vera.pt'

if os.path.exists(sft_drive) and os.path.getsize(sft_drive) > 1_000_000:
    shutil.copy2(sft_drive, sft_local)
    print(f'✓ Copied best_sft_vera.pt from Drive → local disk')
else:
    # Drive write had failed — re-convert from DAgger checkpoint
    print('⚠ Drive SFT file missing/corrupt — re-converting from DAgger checkpoint')
    raw = torch.load(SFT_CKPT, map_location='cpu')
    if isinstance(raw, dict):
        model_sd = raw.get('model_state', raw.get('model', raw))
    else:
        model_sd = raw
    torch.save({'model_state': model_sd}, sft_local)
    print(f'✓ Re-saved DAgger checkpoint → {sft_local}')

# ── Step 4: Patch config to use local output + save_every=5 ───────────────
with open(RL_CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['training']['output_dir'] = LOCAL_OUT
cfg['rl']['save_every']       = 5    # 50 epochs → 10 snapshots locally (~500 MB max)

LOCAL_CONFIG = f'{LOCAL_OUT}/rl_config.yaml'
with open(LOCAL_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

# ── Override globals so Cell 6 picks up the new paths ─────────────────────
RL_CONFIG_PATH = LOCAL_CONFIG
RL_OUT_DIR     = LOCAL_OUT

print(f'\n✓ Drive quota fix applied:')
print(f'  Training output  → {LOCAL_OUT}/rl/')
print(f'  Config           → {LOCAL_CONFIG}')
print(f'  save_every       = 5  (saves ~500 MB locally, not Drive)')
print(f'\nNow run Cell 5b (smoke-test), then Cell 6 (training).')
print(f'After Cell 6 completes, run Cell 6b to copy the best checkpoint to Drive.')

In [ ]:
# ── Cell 6: Run RL training ────────────────────────────────────────────────
# Output streams live — each epoch prints:
#   RL Epoch  N | steps XXXXX | return X.XXXX | success XX.X% | ...
#
# What to watch:
#   success XX.X%  → task success rate (want this to rise above SFT baseline)
#   entropy X.XXXX → keep > 0.5 (if near 0, raise RL_ENTROPY_COEF to 0.08)
#   kl X.XXXX      → keep < 0.5 (if > 1.0, raise RL_KL_COEF to 0.4)
#
# Best checkpoint → {RL_OUT_DIR}/best_rl_vera.pt  (auto-saved every epoch)

import subprocess, sys, os

cmd = [
    sys.executable, '-m', 'training.rl_trainer_vera',
    '--config', RL_CONFIG_PATH,
]

print(f'Starting RL training (seed={SEED}, {RL_EPOCHS} epochs × {RL_NUM_ROLLOUTS} rollouts)')
print(f'SFT checkpoint: {SFT_CKPT}')
print(f'Output dir:     {RL_OUT_DIR}')
print('─' * 70)

proc = subprocess.Popen(
    cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print('\n' + '─' * 70)
    print('✓ RL training complete!')
    print(f'  Best checkpoint: {RL_OUT_DIR}/best_rl_vera.pt')
    print(f'  Full log:        {RL_OUT_DIR}/rl_vera_log.json')
    print(f'  CSV for plots:   {RL_OUT_DIR}/sample_efficiency.csv')
else:
    print(f'\n✗ RL training exited with code {proc.returncode}')
    print('Scroll up for the error.')

In [ ]:
# ── Cell 6b: Copy best checkpoint to Drive after training ───────────────────
# Run after Cell 6 completes. Copies ONLY the best-SR checkpoint and log
# (~50-100 MB total) to Drive. Does NOT copy epoch snapshots.
import os, shutil, torch

# Best SR checkpoint (the one we care about for the paper)
src_sr  = f'{RL_OUT_DIR}/rl/best_sr_vera.pt'
# Store it with a unique name so it doesn't overwrite previous rounds
ROUND = 'r5'   # update this if you run more rounds
dst_sr  = f'{DAGGER_CKPT_DIR}/rl/rl/best_sr_vera_{ROUND}.pt'

if os.path.exists(src_sr):
    meta = torch.load(src_sr, map_location='cpu')
    sr   = meta.get('success_rate', '?')
    ep   = meta.get('epoch', '?')
    sz   = os.path.getsize(src_sr) / 1e6
    print(f'best_sr_vera.pt — epoch={ep}, SR={sr if isinstance(sr, str) else f"{sr:.1%}"},  {sz:.0f} MB')
    try:
        shutil.copy2(src_sr, dst_sr)
        print(f'✓ Saved to Drive: {dst_sr}')
    except OSError as e:
        print(f'✗ Drive still full ({e})')
        print(f'  Download manually from Files panel: {src_sr}')
else:
    print('No best_sr_vera.pt — RL did not improve on DAgger baseline.')
    print('The DAgger checkpoint (best_v3.pt) remains the best policy.')

# Log file (tiny — ~50 KB)
src_log = f'{RL_OUT_DIR}/rl/rl_vera_log.json'
dst_log = f'{DAGGER_CKPT_DIR}/rl/rl/rl_vera_log_{ROUND}.json'
if os.path.exists(src_log):
    try:
        shutil.copy2(src_log, dst_log)
        print(f'✓ Log saved to Drive: {dst_log}')
    except OSError:
        print(f'  Log at: {src_log}')

print('\nDone. Update RL_CONFIG_PATH and RL_OUT_DIR in Cell 7 if you want to plot.')
print(f'  LOG_PATH for Cell 7: {src_log}')

In [ ]:
# ── Cell 7: Plot results ───────────────────────────────────────────────────
import json, os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

LOG_PATH = f'{RL_OUT_DIR}/rl_vera_log.json'

if not os.path.exists(LOG_PATH):
    print(f'Log not found at {LOG_PATH} — run Cell 6 first.')
else:
    with open(LOG_PATH) as f:
        log = json.load(f)

    epochs  = [r['epoch']            for r in log]
    sr      = [r['success_rate']*100 for r in log]
    ret     = [r['mean_return']      for r in log]
    entropy = [r['entropy']          for r in log]
    kl      = [r['kl_loss']          for r in log]

    # Smooth SR with 5-epoch rolling average for readability
    sr_smooth = np.convolve(sr, np.ones(5)/5, mode='same')

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.suptitle(f'VERA RL Training — seed {SEED}', fontsize=13, fontweight='bold')

    # ── Success Rate ──
    ax = axes[0, 0]
    ax.plot(epochs, sr, color='#93c5fd', linewidth=1.0, alpha=0.5, label='raw (per 8 rollouts)')
    ax.plot(epochs, sr_smooth, color='#2563eb', linewidth=2.0, label='5-ep average')
    ax.axhline(10, color='#9ca3af', linewidth=1.2, linestyle='--', label='SFT baseline (~10%)')
    best_sr = max(sr); best_ep = epochs[sr.index(best_sr)]
    ax.axvline(best_ep, color='#f59e0b', linewidth=1, linestyle=':')
    ax.set_title('Task Success Rate (%)', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Success Rate (%)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Mean Return ──
    ax = axes[0, 1]
    ax.plot(epochs, ret, color='#16a34a', linewidth=1.8)
    ax.axhline(max(ret), color='#bbf7d0', linewidth=1, linestyle='--',
               label=f'best={max(ret):.3f} (ep{epochs[ret.index(max(ret))]})')
    ax.set_title('Mean Episode Return', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Return')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Entropy ──
    ax = axes[1, 0]
    ax.plot(epochs, entropy, color='#d97706', linewidth=1.8)
    ax.axhline(0.5, color='#ef4444', linewidth=1, linestyle='--', label='Collapse risk < 0.5')
    ax.set_title('Policy Entropy (exploration)', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Entropy')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── KL vs BC ──
    ax = axes[1, 1]
    ax.plot(epochs, kl, color='#7c3aed', linewidth=1.8)
    ax.axhline(0.5, color='#ef4444', linewidth=1, linestyle='--', label='Forgetting risk > 0.5')
    ax.set_title('KL Divergence vs SFT Anchor', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('KL Loss')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    PLOT_PATH = f'{RL_OUT_DIR}/training_curves.png'
    plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved to {PLOT_PATH}')

    print(f'\nSummary (seed {SEED}):')
    print(f'  SFT baseline : ~10%')
    print(f'  Best SR      : {best_sr:.1f}% (epoch {best_ep})')
    print(f'  Final SR     : {sr[-1]:.1f}% (epoch {epochs[-1]})')
    print(f'  Best return  : {max(ret):.4f} (epoch {epochs[ret.index(max(ret))]})')
    print(f'  Entropy range: {min(entropy):.3f} – {max(entropy):.3f}  (no collapse ✓)')
    print(f'  KL range     : {min(kl):.4f} – {max(kl):.4f}  (BC anchor held ✓)')

## Troubleshooting

| Symptom | Fix |
|---|---|
| `FileNotFoundError: best_sft_vera.pt` | Upload seed123's SFT checkpoint to `VERA_LT_Checkpoints/rl_seed123/best_sft_vera.pt` |
| Success rate stays at 0% for 30+ epochs | Raise `RL_ENTROPY_COEF` to `0.08` in Cell 4, re-run Cell 5+6 |
| Entropy collapses to ~0 | Same fix — more entropy bonus |
| KL explodes > 1.0 | Raise `RL_KL_COEF` to `0.4` |
| CUDA out of memory | Lower `RL_NUM_ROLLOUTS` to `4` |
| Runtime disconnects | Checkpoint auto-saved every epoch — re-run Cell 6, training restarts from scratch but best_rl_vera.pt is safe on Drive |
| `ModuleNotFoundError: language_table` | Cell 3 didn't finish — re-run it |

## Running all 3 seeds in parallel
1. Open 3 separate Colab tabs (File → Open in new tab)
2. In each tab, change `SEED` in Cell 4 to `42`, `123`, `456`
3. Make sure each has its own SFT checkpoint:
   - `rl_seed42/best_sft_vera.pt` (from your seed42 SFT run)
   - `rl_seed123/best_sft_vera.pt` ← **start here first**
   - `rl_seed456/best_sft_vera.pt`